In [0]:
%run ./utils

###Create config and utils

In [0]:
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

try:
    write_log("GOLD_ML_START", "STARTED", "ML comparison started")

    df_curr = spark.read.table("amazon_project.silver.current_clean")
    df_prev = spark.read.table("amazon_project.silver.previous_clean")

    write_log("GOLD_ML_READ", "INFO", "Loaded Silver tables")

    feature_cols = ["price", "rating","discount_percent","quantity_sold"]
    target_col = "total_revenue"

    assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

    models = [
        ("Linear_Regression", LinearRegression(labelCol=target_col, featuresCol="features")),
        ("Decision_Tree", DecisionTreeRegressor(labelCol=target_col, featuresCol="features")),
        ("Random_Forest", RandomForestRegressor(labelCol=target_col, featuresCol="features"))
    ]

    evaluator = RegressionEvaluator(labelCol=target_col, predictionCol="prediction", metricName="r2")

    all_results = []

    for label, df in [("Current", df_curr), ("Previous", df_prev)]:

        train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

        train = assembler.transform(train_data).select("features", target_col)
        test = assembler.transform(test_data).select("features", target_col)

        for name, model_obj in models:
            model = model_obj.fit(train)
            predictions = model.transform(test)

            r2_score = evaluator.evaluate(predictions)

            all_results.append((label, name, float(r2_score)))

    write_log("GOLD_ML_MODELING", "INFO", "Model training and evaluation completed")

    comparison_df = spark.createDataFrame(
        all_results,
        ["dataset", "model_name", "r2_score"]
    )

    comparison_df.write.mode("overwrite").saveAsTable(
        "amazon_project.gold.ml_model_comparison"
    )

    write_log("GOLD_ML_END", "SUCCESS", "ML comparison completed successfully")

except Exception as e:
    write_log("GOLD_ML_ERROR", "FAILED", str(e))
    raise

GOLD_ML_START - STARTED: ML comparison started
GOLD_ML_READ - INFO: Loaded Silver tables
GOLD_ML_MODELING - INFO: Model training and evaluation completed
GOLD_ML_END - SUCCESS: ML comparison completed successfully


In [0]:
display(comparison_df)

dataset,model_name,r2_score
Current,Linear_Regression,0.832172263712456
Current,Decision_Tree,0.9077194951089074
Current,Random_Forest,0.9086181986769791
Previous,Linear_Regression,0.8768676152241617
Previous,Decision_Tree,0.9624834557976442
Previous,Random_Forest,0.9626386761852993
